# 03 — Inference dan submission
Gunakan checkpoint terbaik dan preprocessing yang sama dengan training. Prediksi dipetakan berdasarkan ID, kemudian diurutkan mengikuti sample submission. Nilai nol di sample submission hanyalah contoh format.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd() if (Path.cwd() / 'scripts').is_dir() else Path.cwd().parent
assert (ROOT / 'scripts').is_dir(), 'Buka notebook dari root proyek atau notebooks/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.preprocess import data_directory, read_metadata
DATA = data_directory()
train, test, sample = read_metadata(DATA)

## Prediksi test
Checkpoint baseline menggunakan training split saja; model belum di-refit pada seluruh label. Jika submission sudah tersedia, baca hasilnya. Ubah `REGENERATE` menjadi True untuk inferensi ulang.

In [2]:
from scripts.predict import generate_submission, validate_submission
CHECKPOINT = ROOT / 'checkpoint/baseline_cnn/best.pt'
OUTPUT = ROOT / 'submissions/submission_baseline.csv'
REGENERATE = False
if REGENERATE or not OUTPUT.exists():
    assert CHECKPOINT.exists(), 'Jalankan notebook modeling atau scripts.train terlebih dahulu'
    submission = generate_submission(CHECKPOINT, OUTPUT, DATA)
else:
    submission = pd.read_csv(OUTPUT)
display(submission.head())

,id,label
0,test_00001,0
1,test_00002,0
2,test_00003,6
3,test_00004,10
4,test_00005,13


## Validasi kontrak submission
Harus tepat dua kolom: `id,label`; satu baris per ID, tanpa nilai kosong, dengan label integer 0–15. Urutan harus sesuai template.

In [3]:
validate_submission(submission, sample)
assert len(submission) == len(test)
print(f'Valid: {len(submission):,} baris, kolom {submission.columns.tolist()}')
display(submission.label.value_counts().sort_index().rename('predicted_count').to_frame())
print(OUTPUT)

Valid: 7,994 baris, kolom ['id', 'label']


,predicted_count
label,
0,530
1,483
2,519
3,589
4,495
5,529
6,482
7,530
8,484


C:\ITS\Semester_VII\Supervised Machine Learning\BMKG\New folder\submissions\submission_baseline.csv


## Provenance dan pencatatan leaderboard
Hash SHA-256 mengaitkan submission dengan checkpoint. Skor validasi lokal bukan skor leaderboard. Upload dilakukan manual melalui [halaman kompetisi](https://www.kaggle.com/competitions/document-image-classification-sd234504/data) menggunakan akun yang memiliki akses. Metrik resmi belum dapat diverifikasi karena halaman tidak tersedia pada sesi pemeriksaan.

In [4]:
import hashlib
provenance_path = ROOT / 'reports/submission_baseline_provenance.json'
provenance = json.loads(provenance_path.read_text())
assert hashlib.sha256(OUTPUT.read_bytes()).hexdigest() == provenance['submission_sha256']
display(provenance)

{'checkpoint': 'checkpoint\\baseline_cnn\\best.pt',
 'checkpoint_sha256': 'b16bae99cf0de36e1ed726824ec879d8935d62d8bbb7eb0d7157cd2d92a32a46',
 'submission_sha256': '5fd9913aedf081fdb66e9823eb28e36e6146162113f3d1640f87be76fd4f6d5c',
 'rows': 7994,
 'label_counts': {'0': 530,
  '1': 483,
  '2': 519,
  '3': 589,
  '4': 495,
  '5': 529,
  '6': 482,
  '7': 530,
  '8': 484,
  '9': 452,
  '10': 610,
  '11': 382,
  '12': 498,
  '13': 552,
  '14': 398,
  '15': 461},
 'best_epoch': 5,
 'validation_metrics': {'val_loss': 1.0758372708724009,
  'accuracy': 0.6803752931978108,
  'macro_f1': 0.6730166243992567},
 'public_score': None,
 'private_score': None}

## Checklist pelaporan
Catat nama file, run, epoch terbaik, skor validasi, dan skor resmi setelah submission. Dataset, cache, checkpoint, dan submission lokal diabaikan oleh `.gitignore`. Commit source, notebook yang sudah dieksekusi, konfigurasi, dan laporan kecil. Jalankan `python -m scripts.validate_repository` sebelum mengunggah repository.